# 03 - Forecasting V3 (Weather Features + True t+1 Forecasting)

This notebook trains AQI(t+1) forecasting models using pollution, temporal, and weather features from v3 preprocessing.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

sns.set_theme(style='whitegrid')

TRAIN_PATH = Path('data/splits/train_v3.csv')
TEST_PATH = Path('data/splits/test_v3.csv')
FEATURES_PATH = Path('outputs/models/feature_columns_v3.json')
SCALER_PATH = Path('outputs/models/scaler_v3.pkl')

REPORTS_DIR = Path('outputs/reports')
MODELS_DIR = Path('outputs/models')
FIG_DIR = Path('outputs/figures')
for directory in [REPORTS_DIR, MODELS_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


## Step 1 - Load Data

In [ ]:
for path in [TRAIN_PATH, TEST_PATH, FEATURES_PATH, SCALER_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
feature_columns = json.loads(FEATURES_PATH.read_text(encoding='utf-8'))
scaler = joblib.load(SCALER_PATH)

required_cols = ['Timestamp', 'target_timestamp', 'City', 'AQI'] + feature_columns
missing_train = [col for col in required_cols if col not in train_df.columns]
missing_test = [col for col in required_cols if col not in test_df.columns]
if missing_train:
    raise ValueError(f'Missing train columns: {missing_train}')
if missing_test:
    raise ValueError(f'Missing test columns: {missing_test}')

train_df['Timestamp'] = pd.to_datetime(train_df['Timestamp'])
test_df['Timestamp'] = pd.to_datetime(test_df['Timestamp'])
train_df['target_timestamp'] = pd.to_datetime(train_df['target_timestamp'])
test_df['target_timestamp'] = pd.to_datetime(test_df['target_timestamp'])

X_train = train_df[feature_columns].astype(float)
y_train = train_df['AQI'].astype(float)
X_test = test_df[feature_columns].astype(float)
y_test = test_df['AQI'].astype(float)

assert X_train.isna().sum().sum() == 0, 'NaNs in X_train.'
assert X_test.isna().sum().sum() == 0, 'NaNs in X_test.'
assert y_train.isna().sum() == 0, 'NaNs in y_train.'
assert y_test.isna().sum() == 0, 'NaNs in y_test.'
assert train_df['target_timestamp'].max() < test_df['target_timestamp'].min(), 'Train/test target overlap.'
assert (train_df['Timestamp'] < train_df['target_timestamp']).all(), 'Train timestamp leakage.'
assert (test_df['Timestamp'] < test_df['target_timestamp']).all(), 'Test timestamp leakage.'

print('Train shape:', X_train.shape)
print('Test shape :', X_test.shape)
print('Feature count:', len(feature_columns))


## Step 2 - Naive Baseline (AQI_lag1 In Original Units)

In [ ]:
def inverse_scaled_feature(values, feature_name):
    if feature_name not in feature_columns:
        raise ValueError(f'{feature_name} not present in feature list.')
    idx = feature_columns.index(feature_name)
    return np.asarray(values, dtype=float) * scaler.scale_[idx] + scaler.mean_[idx]

y_pred_naive = inverse_scaled_feature(test_df['AQI_lag1'].values, 'AQI_lag1')
print('Naive baseline uses AQI_lag1 converted back to original AQI units.')
print('Naive prediction min/max:', float(np.min(y_pred_naive)), float(np.max(y_pred_naive)))


## Step 4 - Metrics

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true_arr = np.asarray(y_true, dtype=float)
    y_pred_arr = np.asarray(y_pred, dtype=float)
    return {
        'MAE': mean_absolute_error(y_true_arr, y_pred_arr),
        'RMSE': np.sqrt(mean_squared_error(y_true_arr, y_pred_arr)),
        'MAPE': np.mean(np.abs((y_true_arr - y_pred_arr) / y_true_arr)) * 100.0,
    }

results = []
predictions = {}

metrics_naive = compute_metrics(y_test, y_pred_naive)
results.append({'Model': 'Naive Baseline', **metrics_naive})
predictions['Naive Baseline'] = y_pred_naive
metrics_naive


## Step 3 - Train Models

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
pred_linear = linear_model.predict(X_test)
results.append({'Model': 'Linear Regression', **compute_metrics(y_test, pred_linear)})
predictions['Linear Regression'] = pred_linear

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)
try:
    rf_model.fit(X_train, y_train)
except PermissionError:
    rf_model = RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=1,
    )
    rf_model.fit(X_train, y_train)
pred_rf = rf_model.predict(X_test)
results.append({'Model': 'Random Forest', **compute_metrics(y_test, pred_rf)})
predictions['Random Forest'] = pred_rf

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1,
)
try:
    xgb_model.fit(X_train, y_train)
except PermissionError:
    xgb_model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective='reg:squarederror',
        n_jobs=1,
    )
    xgb_model.fit(X_train, y_train)
pred_xgb = xgb_model.predict(X_test)
results.append({'Model': 'XGBoost', **compute_metrics(y_test, pred_xgb)})
predictions['XGBoost'] = pred_xgb

print('Models trained and evaluated.')


## Steps 5-6 - Selection And Save Outputs

In [ ]:
comparison = pd.DataFrame(results).sort_values(['MAE', 'RMSE', 'MAPE']).reset_index(drop=True)
best_model_name = comparison.loc[0, 'Model']

model_map = {
    'Linear Regression': linear_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
}

comparison.to_csv(REPORTS_DIR / 'model_comparison_v3.csv', index=False)
if best_model_name == 'Naive Baseline':
    joblib.dump({'type': 'naive_baseline', 'feature': 'AQI_lag1', 'scaler': 'scaler_v3.pkl'}, MODELS_DIR / 'best_model_v3.pkl')
else:
    joblib.dump(model_map[best_model_name], MODELS_DIR / 'best_model_v3.pkl')
(MODELS_DIR / 'best_model_name_v3.txt').write_text(best_model_name, encoding='utf-8')

display(comparison)
print('Best model:', best_model_name)
print('Saved:', REPORTS_DIR / 'model_comparison_v3.csv')
print('Saved:', MODELS_DIR / 'best_model_v3.pkl')
print('Saved:', MODELS_DIR / 'best_model_name_v3.txt')


## Step 7 - Visualizations

In [ ]:
best_pred = predictions[best_model_name]
eval_df = test_df[['Timestamp', 'target_timestamp', 'City']].copy()
eval_df['actual'] = y_test.values
eval_df['predicted'] = best_pred
eval_df['residual'] = eval_df['actual'] - eval_df['predicted']
eval_df['abs_error'] = eval_df['residual'].abs()
eval_df['month'] = eval_df['Timestamp'].dt.month

daily_eval = eval_df.groupby('target_timestamp', as_index=False)[['actual', 'predicted']].mean().head(180)
plt.figure(figsize=(14, 6))
plt.plot(daily_eval['target_timestamp'], daily_eval['actual'], label='Actual AQI(t+1)', linewidth=2)
plt.plot(daily_eval['target_timestamp'], daily_eval['predicted'], label='Predicted AQI(t+1)', linewidth=2)
plt.title(f'V3 Predicted vs Actual AQI - {best_model_name}')
plt.xlabel('Target Date')
plt.ylabel('AQI')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'predicted_vs_actual_v3.png', dpi=200)
plt.close()

plt.figure(figsize=(12, 6))
sns.histplot(eval_df['residual'], bins=60, kde=True)
plt.title(f'V3 Residual Distribution - {best_model_name}')
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIG_DIR / 'residuals_distribution_v3.png', dpi=200)
plt.close()

plt.figure(figsize=(12, 6))
sns.boxplot(data=eval_df, x='month', y='abs_error', color='steelblue')
plt.title(f'V3 Absolute Error by Feature Month - {best_model_name}')
plt.xlabel('Month')
plt.ylabel('Absolute Error')
plt.tight_layout()
plt.savefig(FIG_DIR / 'error_by_month_v3.png', dpi=200)
plt.close()

city_order = eval_df.groupby('City')['abs_error'].median().sort_values(ascending=False).index
plt.figure(figsize=(14, 6))
sns.boxplot(data=eval_df, x='City', y='abs_error', order=city_order)
plt.title(f'V3 Absolute Error by City - {best_model_name}')
plt.xlabel('City')
plt.ylabel('Absolute Error')
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(FIG_DIR / 'error_by_city_v3.png', dpi=200)
plt.close()

print('Saved v3 plots.')


## Step 8 - Summary

In [ ]:
best = comparison.iloc[0]
second = comparison.iloc[1]
print('Metrics table:')
display(comparison)
print('Best model:', best_model_name)
print('Key observations:')
print(f"- Best MAE: {best['MAE']:.3f} AQI points from {best_model_name}.")
print(f"- Next best by MAE: {second['Model']} at {second['MAE']:.3f} AQI points.")
print(f"- Weather-enhanced feature count: {len(feature_columns)}.")
